In [ ]:
import os, base64, json, re, tqdm
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from glob import glob
from skimage import io
import feret

import HFcluster, Split, CellNeighborhood, CI

# color palette file
xkcd_cols = HFcluster.load_xkcd()

# generate label masks from Biodock results (GeoJSON)

In [ ]:
from shapely.geometry import Polygon, MultiPolygon
from skimage.draw import polygon as draw_poly

In [ ]:
###########INPUTS############
# initialize processed CODEX directory
exp_dir = r'N:\CODEX processed\XXXXXX' #processed data folder
# which z slice to quantify
use_z = 0

# initialize directory of mask data from Biodock
mask_dir = r'N:\CODEX analysis\XXXXXX\biodock'

# enumerate class types from biodock
# make sure that the spelling matches exactly that of Biodock (case sensitive!)
classes = {'none':0, 'Myofiber':1, 'Motor nerve':2, 'Capillary':3, 'Muscle spindle':4, 'Smooth Muscle':5, 'Regenerating myofiber':6}

# experiment parameters
regs = 5
gx = 7 # x tiles
gy = 5 # y tiles
b = 0 # border pixels trimmed (default 0)
scale=0.37744
#############################

# find mask json files
files = sorted(glob(mask_dir+'/*.json'))
#index = glob(mask_dir+'/index.json')

# create output directory
man_mask_d = os.path.join(mask_dir,'RGBmask')
if not os.path.exists(man_mask_d):
    os.mkdir(man_mask_d)

run = exp_dir.split('\\')[-1]

# pointing to stitched image directory
mosaic_dir = os.path.join(exp_dir,'postprocessed')

# getting channel names from channelnames.txt
chnames = pd.read_csv(os.path.join(exp_dir,'channelnames.txt'), header=None)[0].values

# solving CODEX experiment information
nch = len(chnames)
assert len(chnames)%4==0
ncyc = len(chnames)//4

#cy_ch = chnames.reshape(ncyc,4)

In [ ]:
for r in range(regs):
    r += 1
    print(f'reg{r:03d}')
    file = os.path.join(mask_dir,f'{r:01d}.json')
    
    with open(file) as maskfile:
        mask_results = json.load(maskfile)
        region_img = os.path.join(exp_dir,'stitched',f'mosaic_region{r:03d}_z{use_z:02d}_cy01_ch01.tif')
        assert os.path.exists(region_img)
        hc,wc=io.imread(region_img).shape

        mask = np.zeros((hc,wc), dtype="uint8")
        id_mask = np.zeros((hc,wc), dtype="uint32")
        objects_encoded = mask_results['features']

        # getting trim parameters
        htrim = 0
        while (hc-htrim-2*b)%gy!=0 or (hc-htrim-2*b)/gy%2!=0:
            htrim+=1

        wtrim = 0
        while (wc-wtrim-2*b)%gx!=0 or (wc-wtrim-2*b)/gx%2!=0:
            wtrim+=1

        CellSeg_cache_file = f"{run}_reg{r:03d}.npz"

        df = pd.DataFrame(columns=['obj_id','class','class_id','area','min_feret'])
        for obj in tqdm.tqdm(objects_encoded):
            oci = classes[obj['properties']['name']]
            
            if len(obj['geometry']['coordinates']) > 1:
                continue
            p = Polygon(obj['geometry']['coordinates'][0])
            if p.area==0: continue
            poly_coordinates = np.array(list(p.exterior.coords))
            cc,rr = draw_poly(poly_coordinates[:,1], poly_coordinates[:,0], mask.shape)
            if np.sum(mask[cc,rr])>0: continue
            mask[cc,rr] = oci
            id_mask[cc,rr] = obj['properties']['id']

            df.loc[obj['properties']['id']]=[obj['properties']['id'],obj['properties']['name'],oci,p.area,feret.min(id_mask==obj['properties']['id'])]

        print('saving')
        io.imsave(f'{man_mask_d}/region{r:03d}_class.png', mask.astype(np.uint8), check_contrast=False)
        io.imsave(f'{man_mask_d}/region{r:03d}_id.tif', id_mask.astype(np.uint32), check_contrast=False)
        np.savez_compressed(os.path.join(man_mask_d, CellSeg_cache_file), data=id_mask[htrim//2:id_mask.shape[0]-(htrim-htrim//2),wtrim//2:id_mask.shape[1]-(wtrim-wtrim//2)].astype(np.uint32))
        plt.imsave(f'{man_mask_d}/region{r:03d}.png', mask, cmap='gnuplot')
        df.to_csv(f'{man_mask_d}/region{r:03d}.csv')        

## STOP HERE: RUN CELLSEG WITH NUCLEI AND BIODOCK MASKS THEN CONTINUE

####Labeled mask quantification
1. copy paste CSQuant_config.py into the processed folder.
    -> CSQuant.py will integrate the masks from Biodock + nuclear segmentation and create a segm-mask folder
2. Modify CSQuant_config.py with the growth parameters from CellSeg
3. In the terminal, go to c:
4. conda activate CellSeg11
5. cd Documents\Github\CSQuant\
6. python run_quant.py

####Nuclei segmentation and quantification
1. copy and paste CellSeg_config.py into  the processed folder.
2. Modify CellSeg_config.py with the growth parameters from CellSeg
3. In the terminal, go to c:
4. conda activate CellSeg11
5. cd Documents\Github\CellSeg-CRISP\
6. python run_cellSeg-CRISP.py

In [ ]:
#Integration with quant and nuclei data
nuc_segm = 'segm-1'
nn_segm = 'segm-mask'
quant_type = 'loose'
###########################################
segm_dir = os.path.join(exp_dir,'processed','segm')
nuc_dir = os.path.join(segm_dir,nuc_segm,'fcs',quant_type)
nn_dir = os.path.join(segm_dir,nn_segm,'fcs',quant_type)

In [ ]:
for r in range(regs):
    r=r+1
    print('merging quantifications')
    nn_morph = pd.read_csv(f'{man_mask_d}/region{r:03d}.csv', index_col=0)
    
    nn_f = glob(os.path.join(nn_dir,f'*_region{r:03d}*_fib_*_loose.csv'))
    nuc_f = glob(os.path.join(nuc_dir,f'*_reg{r:03d}*_fib_*_loose.csv'))
    
    assert len(nn_f)==1, AssertionError(f'CSV files found... {nn_f}')
    assert len(nuc_f)==1, AssertionError(f'CSV files found... {nuc_f}')

    nn_quant = pd.read_csv(nn_f[0],index_col='cell_id:cell_id')
    nn_quant.index = nn_quant.index.astype(int)

    nn_quant = pd.concat([nn_quant,nn_morph],axis=1)
    nn_quant = nn_quant[nn_quant['size:size']!=0]

    nuc_quant = pd.read_csv(nuc_f[0],index_col='cell_id:cell_id')

    def getmaskvalues(cells_df, mask, x='x:x', y='y:y'):
        xs = np.clip(np.rint(cells_df.loc[:,x].to_numpy()).astype(np.int32), 0, mask.shape[1]-1)
        ys = np.clip(np.rint(cells_df.loc[:,y].to_numpy()).astype(np.int32), 0, mask.shape[0]-1)
        return mask[(ys, xs)]
    
    print(f"loading labeled mask at {man_mask_d}/region{r:03d}_id.tif")
    label_mask = io.imread(f'{man_mask_d}/region{r:03d}_id.tif')

    nuc_quant['nn_obj_id'] = getmaskvalues(nuc_quant, label_mask)

    nuc_quant['nn_class'] = ['none' if i==0 or i not in nn_quant.index else nn_quant.loc[i,'class'] for i in nuc_quant['nn_obj_id']]
    nuc_quant['nn_class_id'] = [classes[i] for i in nuc_quant['nn_class']]
    
    nuc_quant.to_csv(os.path.join(nuc_dir,f'{run}_reg{r:03d}_nn_merge.csv'))
    nn_quant.to_csv(os.path.join(nn_dir,f'{run}_reg{r:03d}_nn_merge.csv'))

## Load Fiber quantifications for clustering and annotation

In [ ]:
dfs = []
for r in range(regs):
    r=r+1
    print(f'loading nn quants for reg{r:03d}')

    assert len(glob(f'{nn_dir}/{run}_reg{r:03d}_nn_merge.csv'))==1
    nn_quant = pd.read_csv(glob(f'{nn_dir}/{run}_reg{r:03d}_nn_merge.csv')[0],index_col=0)
    nn_quant.index = nn_quant.index.astype(int)

    nn_quant = nn_quant[nn_quant['size:size']!=0]
    
    dfs += [nn_quant]
dfs = pd.concat(dfs)

dfs = dfs.reset_index(names='reg_cell_id')
dfs = dfs.dropna(axis=0)

In [ ]:
import scanpy as sc, anndata as ad
import seaborn as sns

In [ ]:
# make sure that the entries in cond_dict match region number
cond_dict = {1:'noTumor', 2:'preCachexia', 3:'Cachexia', 4: 'day3recovery', 5: 'day7recovery'}
muscle_dict = {1:'GA', 2:'TA+EDL', 3:'TA+EDL', 4: 'TA+EDL', 5: 'TA+EDL'}

In [ ]:
#data = sc.read_h5ad('nn_merged.h5ad')

In [ ]:
data = ad.AnnData(dfs.loc[:,dfs.columns.str.startswith('cyc')])
data.obs_names_make_unique()
data.obs = dfs.loc[:,~dfs.columns.str.startswith('cyc')]
data.obs.columns = [c.split(':')[0] for c in data.obs.columns]

# assigns a condition column based on cond_dict
# make sure that the entries in cond_dict match region number
data.obs['condition'] = [cond_dict[i] for i in dfs['region:region']]

var_df=pd.Series(data.var.index).str.split(':',expand=True)

data.var['cycle'] = var_df[0].str.split('_',expand=True)[0].str[-3:].astype(int).values
data.var['channel'] = var_df[0].str.split('_',expand=True)[1].str[-4:-1].astype(int).values
data.var['quant_type'] = var_df[0].str.split('_',expand=True)[1].str[-1].values
data.var['marker'] = var_df[1].str.split('_',expand=True)[0].values

data.obs['min_feret_um'] = data.obs['min_feret']*scale
data.obs['area_um'] = data.obs['area']*(scale**2)

In [ ]:
data.obs['shift_x']=[(i-1)*14000 for i in data.obs['region']]+data.obs['x']
#data.obs['shift_y']=[(i-1)*14000 for i in data.obs['run_id']]+data.obs['y'] ## for multi-run analysis

In [ ]:
sns.set(font_scale=1, rc={'figure.figsize':(4,4)})
sns.set_style("white")
sc.plotting.violin(data[data.obs['class']=='Myofiber'],
                   groupby='condition',keys=['min_feret_um'], 
                   order=['Uninjured','day1','day3','day7'], use_raw=False,
                   multi_panel=True)

In [ ]:
from scipy import stats

In [ ]:
df = data.obs[data.obs['class']=='Myofiber']

In [ ]:
s1 = df[df['condition']=='Uninjured']['min_feret_um']
s2 = df[df['condition']=='day7']['min_feret_um']
print(stats.ranksums(s1,s2))

print(s1.describe())
print(s2.describe())

In [ ]:
sns.lineplot(data.obs[data.obs['class']=='Myofiber'],x='region',y='area_um')
plt.xlim(2,5)
plt.ylim(0,2200)

In [ ]:
clust_markers = [
 'cyc002_ch002f:A',
 'cyc019_ch002f:B',
 'cyc019_ch003f:C',
 'cyc019_ch004f:D',
 ...]

In [ ]:
clustering = data[:,clust_markers]

In [ ]:
sc.pp.log1p(clustering)
sc.pp.scale(clustering, max_value=10)

In [ ]:
sc.tl.pca(clustering, svd_solver="arpack")

sns.set(rc={'figure.figsize':(4,4)}, style='white')
sc.pl.pca(clustering, color=['region'], vmin=1, vmax=5, cmap='hsv')

In [ ]:
sc.pl.pca_variance_ratio(clustering, log=True)

In [ ]:
sc.pp.neighbors(clustering, n_neighbors=40, n_pcs=None)
sc.tl.umap(clustering, min_dist=0.1)

In [ ]:
sc.tl.leiden(
    clustering,
    resolution=1,
    random_state=42,
    n_iterations=5,
    directed=False,
)

In [ ]:
sns.set(rc={'figure.figsize':(4,4)}, style='white')
sc.pl.umap(clustering, color=clust_markers, vmin=0, vmax=3)

In [ ]:
data.obs = data.obs.drop('leiden',axis=1)
data = HFcluster.transfer_meta(data, clustering)

data.obsm['spatial'] = np.array(list(zip(data.obs['x'].values,data.obs['y'].values)))
data.obsm['tissue_array'] = np.array(list(zip(data.obs['shift_x'],data.obs['y'].values)))

In [ ]:
sns.set(rc={'figure.figsize':(6,6)}, style='white')
sc.pl.umap(data, color='leiden', alpha=0.5,legend_loc='on data')
plt.savefig('leiden.svg')

In [ ]:
sns.set(rc={'figure.figsize':(36,12)}, style='white')
sns.scatterplot(data=data.obs,x='shift_x',y='y',hue='class',size='area_um',
                sizes=(10, 500),linewidth=0.1,edgecolor='Black',palette=data.uns['leiden_colors'])
plt.ylim(6000,-500)
plt.xlim(-1000,53000)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
plt.savefig('myofiber_class.svg')

In [ ]:
sns.set(rc={'figure.figsize':(36,12)}, style='white')
sns.scatterplot(data=data.obs,x='shift_x',y='y',hue='leiden',size='area_um',
                sizes=(10, 200),linewidth=0.1,edgecolor='Black',palette=data.uns['leiden_colors'])
plt.ylim(6000,-500)
plt.xlim(-1000,53000)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
plt.savefig('tissue_leiden.svg')

In [ ]:
ch = 'cyc008_ch004b:Atrogin1_border'
df = data.obs
df[ch] = np.array(data[:,ch].X)
sns.set(rc={'figure.figsize':(36,12)}, style='white')
sns.scatterplot(data=df[df['class']=='Myofiber'],x='shift_x',y='y',hue=ch,size='area_um',
                sizes=(10, 200),linewidth=0.1,edgecolor='Black', palette='Reds')
plt.ylim(6000,-500)
plt.xlim(-1000,53000)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
#plt.savefig('tissue_leiden.svg')

In [ ]:
sns.set(rc={'figure.figsize':(6,6)}, style='white')
sc.pl.umap(data, color=['leiden','class'], alpha=0.5, legend_loc='on data')
plt.savefig('leiden.svg')

In [ ]:
sns.set(font_scale=0.5)
sc.pl.heatmap(clustering, clust_markers+['class_id'], groupby='leiden',swap_axes=False,log=False,vmin=0, vmax=6, cmap='cubehelix')

In [ ]:
sns.set(font_scale=2)
sns.set_style("white")
sc.tl.rank_genes_groups(clustering, 'leiden', method='logreg')
sc.pl.rank_genes_groups(clustering, n_genes=10, sharey=False, ncols=4)

In [ ]:
sns.set(font_scale=2)
sns.set_style("white")
sc.tl.rank_genes_groups(clustering, 'class', method='logreg')
sc.pl.rank_genes_groups(clustering, n_genes=10, sharey=False, ncols=4)

In [ ]:
fiber_counts = data.obs.groupby(['region','condition','class']).count()['x'].reset_index()
fiber_counts = fiber_counts[fiber_counts['x']!=0]

fiber_counts.to_csv("fiber_counts.csv")

sns.barplot(fiber_counts, x='condition', y='x', hue='class', errorbar=None, errwidth=1, capsize=0.02, 
            order=['Uninjured','day1','day3','day7'])

plt.xticks(rotation=70)
plt.ylabel('Number of myofibers')
plt.savefig('myofiber_count.svg')

In [ ]:
data_s=data[data.obs['class']=='Myofiber',clust_markers]
sc.tl.rank_genes_groups(data_s, 'condition', reference='Uninjured', method='t-test_overestim_var', corr_method='benjamini-hochberg')

sns.set(rc={'figure.figsize':(4,4)}, style='white')
sc.pl.rank_genes_groups(data_s, n_genes=10)

In [ ]:
## save mask clustering data
data.write("nn_merged.h5ad")

# Nuclear clustering and analysis

In [ ]:
# can be multi-run
runs = ['XXXXXX']
path = r'N:\CODEX processed'

#subdirectory locations of csv files 
subdir = f"processed/segm/segm-1/fcs/{quant_type}/" #default is Akoya CODEX processor output location; use "" if csvs are not in subfolders
filename_pattern = '*reg*_nn_merge.csv' #filename formate of csv files; used * as wildcards 

#output directory
outdir = 'N:/CODEX analysis/XXXXXX/'
#where to save output heatmaps and csvs

In [ ]:
if os.path.isfile('adata.h5ad'):
    adata=sc.read_h5ad('adata.h5ad')

In [ ]:
# Load data
data, data_all = HFcluster.loadData(runs, path, subdir, filename_pattern,
                                    scale=False, noise=400, scale_val=10, q=0.99, center=0.3,
                                    DNAfilter=True, ManualDNAthreshold=10, DNAChannel='cyc002_ch001i:DAPI-02_interior', DRAQ5Channel='cyc020_ch004i:DRAQ5_interior')

In [ ]:
#convert df to adata
adata = HFcluster.conv_adata(data_all)

In [ ]:
sc.pl.scatter(adata, x="DRAQ5_i", y="DAPI-02_i")

In [ ]:
# noise suppression *** only use if not scaled during loading
if False:
    adata.X = adata.X-400
    adata.X[adata.X<0] = 0
    adata.X = np.nan_to_num(adata.X, nan=0, posinf=0, neginf=0)

In [ ]:
adata.obs['condition'] = [cond_dict[i] for i in adata.obs['region']]
adata.obs['muscle group'] = [muscle_dict[i] for i in adata.obs['region']]

In [ ]:
adata = HFcluster.generateTissueArray(adata, shift=14000)

In [ ]:
HFcluster.plotlyTissueArray(adata,'CD45_b', palette=xkcd_cols, log=False, size_max=3)

In [ ]:
# save as h5ad
adata.write('adata.h5ad')

# clustering

In [ ]:
# Print list of marker channels
list(adata.var_names)

In [ ]:
markers =  [
 'MyoD_i',
 'Myogenin_i',
 'Pax7_i',
 ...]

In [ ]:
if False: #log transform
    sc.pp.log1p(adata)

if False: #rescale data
    for r in adata.obs['run'].unique():
        sc.pp.scale(adata, zero_center=False, max_value=10, mask_obs=adata.obs['run']==r)

In [ ]:
adata_sub = adata[:,markers].copy()

In [ ]:
sc.tl.pca(adata_sub, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_sub, log=True, save='.svg')

In [ ]:
sc.tl.pca(adata_sub, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_sub, log=True, save='.svg')
sc.pp.neighbors(adata_sub, n_neighbors=40, use_rep='X', metric='euclidean')
sc.tl.umap(adata_sub, min_dist=0.1, random_state=42)

In [ ]:
sns.set(style='white')
sc.pl.umap(adata_sub,color=['condition'])

In [ ]:
sc.tl.leiden(adata_sub, resolution=1)

In [ ]:
sns.set_theme(style='white')
sc.pl.umap(adata_sub,color=['leiden'], legend_loc='on data')

In [ ]:
sns.set_theme(style='white')
sc.pl.umap(adata_sub,color=['nn_class'], palette='cubehelix', size=20, alpha=0.2)

In [ ]:
sns.set(style='white', font_scale=1, rc={'figure.figsize':(5,4)})
sc.pl.umap(adata_sub,color=sorted(markers), vmin=0, vmax=5000, cmap='Reds')

In [ ]:
sns.set(style='white', font_scale=1)
sc.tl.rank_genes_groups(adata_sub, 'leiden', method='logreg')
sc.pl.rank_genes_groups(adata_sub, n_genes=10, sharey=False, ncols=4)

In [ ]:
adata.obs['leiden_res2'] = adata.obs['leiden']
adata.obs = adata.obs.drop('leiden',axis=1)
adata = HFcluster.transfer_meta(adata, adata_sub)
adata.uns['clust_markers']=markers

In [ ]:
HFcluster.plotlyTissueArray(adata,'leiden', palette=adata.uns['leiden_colors'], log=False, size_max=3)

In [ ]:
order = ['Uninjured','day1','day3','day7']

cluster_props=HFcluster.get_cluster_proportions(adata, cluster_key='leiden', sample_key='condition',prop=True)
cluster_props=cluster_props[order]
cluster_props=cluster_props.T

sns.set(style='white', font_scale=0.7, rc={'figure.figsize':(4,6)})
HFcluster.plot_cluster_proportions(cluster_props, adata.uns['leiden_colors'])
plt.savefig('subset_proportions.svg')

## load annotations

In [ ]:
# Load CSV containing celltype names
adata = HFcluster.loadCelltypeNames(adata, 'leiden2celltypenames.csv', 'leiden')
# Read numerical celltypeIDs as a dictionary
celltypeid2name = pd.read_csv(outdir+'celltypename2id.csv', index_col=1, header=None).to_dict()[0]

In [ ]:
sns.set(style='white',rc={'figure.figsize':(6,6)}, font_scale=1)
sc.pl.umap(adata,color=['celltypename'])

In [ ]:
counts = adata.obs.groupby(['run_region','celltypename']).count()['run'].unstack()
counts = counts.T/(counts.sum(axis=1))

sns.set(font_scale=0.7)
sns.clustermap(counts, method='ward', figsize=(6,8), col_cluster=True, cmap='Purples')
plt.savefig('celltypename_composition.svg')

In [ ]:
[dict(zip(sorted(adata.obs['celltypename'].unique()),adata.uns['celltypename_colors']))[i] for i in adata.obs['celltypename'].unique()]

In [ ]:
HFcluster.plotlyTissueArray(adata,'celltypename', palette=[dict(zip(sorted(adata.obs['celltypename'].unique()),adata.uns['celltypename_colors']))[i] for i in adata.obs['celltypename'].unique()], log=False, size_max=3)

In [ ]:
cluster_props=HFcluster.get_cluster_proportions(adata, cluster_key='celltypename', sample_key='condition',prop=True)
cluster_props=cluster_props[order]
cluster_props=cluster_props.T

sns.set(style='white', font_scale=0.7, rc={'figure.figsize':(4,6)})
HFcluster.plot_cluster_proportions(cluster_props, adata.uns['celltypename_colors'])
plt.savefig('celltypename_proportions.svg')

In [ ]:
counts = adata.obs.groupby(['run_region','region','celltypename']).count()['run'].unstack()
counts = counts[counts.sum(axis=1)!=0]
counts = counts.T/(counts.sum(axis=1))
counts.to_csv('celltype_prop.csv')

for i in counts.index:
    print(i)
    df = counts.loc[i].reset_index()

    sns.set(style='white', font_scale=1, rc={'figure.figsize':(2,4)})
    sns.lineplot(data=df, x='region', y=i)
    plt.xlim(1,5)
    plt.ylim(0)
    plt.savefig(f'subset_{i}_.svg')
    plt.show()

In [ ]:
data_s=adata[adata.obs['celltypename']=='Myofibers',markers]
sc.tl.rank_genes_groups(data_s, 'condition', reference='Uninjured', method='t-test_overestim_var', corr_method='benjamini-hochberg')

sns.set(rc={'figure.figsize':(4,4)}, style='white')
sc.pl.rank_genes_groups(data_s, n_genes=10)

# Neighborhood analysis

In [ ]:
# knn windowing to identify neighbors and mini-kMeans to find n_eighborhoods
adata, niches = CellNeighborhood.find(adata, cluster_col='subset', sample_col='run_region', n_neighbors = 40, k_neighborhoods = 15, drop=False, plot=False)

In [ ]:
# leiden clustering of neighbors into neighborhoods
sns.set(style='white', rc={'figure.figsize':(5,4)})
adata, niches = CellNeighborhood.leiden_cluster(adata, niches, celltype_col='subset', resolution=0.2, cluster_size_filter=500)

In [ ]:
sns.scatterplot(x=adata.obsm['neighborhood_umap_X'], y=adata.obsm['neighborhood_umap_Y'], hue=adata.obs['knn_niche'], s=1)
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

In [ ]:
counts = adata.obs.groupby(['run_region','knn_niche']).count()['run'].unstack()
counts = counts.T/(counts.sum(axis=1))

sns.set(font_scale=1)
sns.clustermap(counts.drop(-1,axis=0), z_score=0, vmin=-2, vmax=2, figsize=(5,8), cmap='bwr', metric='correlation')
plt.savefig('knn_niche_composition.svg')

In [ ]:
adata.obs['knn_niche'] = adata.obs['knn_niche'].astype('category')
HFcluster.plotlyTissueArray(adata,'knn_niche', palette=cols, size_max=4, size=[10]*len(adata), height=800, width=1000, template='simple_white')

In [ ]:
# calculate celltype enrichment within each neighborhood
neighbor_enrichment = CellNeighborhood.neighborhood_enrichment(adata[adata.obs['knn_niche']!=-1].obs, 'knn_niche', 'subset_2', default_col='run', vmin =-2,vmax = 2, metric="correlation", method="ward", col_cluster = True, row_cluster = True, figsize = (14,10))
plt.savefig('knn_niche_enrichments.svg')

In [ ]:
niche_dict={'0':...}

In [ ]:
adata.obs['neighborhood']=[niche_dict[i] for i in adata.obs['knn_niche']]

In [ ]:
HFcluster.plotlyTissueArray(adata,'neighborhood', palette=xkcd_cols[50:], size_max=5, size=[10]*len(adata), height=800, width=1000, template='simple_white')

In [ ]:
counts = adata.obs.groupby(['run_region','condition','neighborhood']).count()['run'].unstack()
counts = counts[counts.sum(axis=1)!=0]
counts = counts.T/(counts.sum(axis=1))
counts.to_csv('neighborhood_prop.csv')

for i in counts.index:
    print(i)
    df = counts.loc[i].reset_index()

    sns.set(style='white', font_scale=1.4, rc={'figure.figsize':(2,4)})
    sns.barplot(data=df, x='condition', y=i, order=['WT','PSGL1-KO'], legend=False, errorbar='se', capsize=0.15, hue='condition', palette='Set1')
    sns.swarmplot(data=df, x='condition', y=i, order=['WT','PSGL1-KO'], legend=False, color='Black')
    plt.savefig(f'neighorhood_{i}.svg')
    plt.show()

# Export for further analysis

In [ ]:
for i in ['reg_cellID', 'cell_id', 'region', 'tile_num', 'x','y','z','x_tile','y_tile','size','interior','border','array_x','array_y']:
    adata.obs[i] = adata.obs[i].astype(int)
for i in ['celltype_id', 'kmeans_neighborhood', 'knn_niche', 'leiden']:
    adata.obs[i] = adata.obs[i].astype(str)

# save as h5ad
adata.write('adata_filtered.h5ad')

# Cell-cell interaction analysis

In [ ]:
import CI
# create subdirectory for niche analysis
nichedir = outdir+'niche_analysis/'
HFcluster.newDir(nichedir)

In [ ]:
# append tissue treatment condition to each cell niche
niches['condition'] = [tissuedict[int(i[-1])] for i in niches['run_region']]

In [ ]:
circle_order = [...]

In [ ]:
# plot cell-cell interaction enrichment
col_name='subset'   
condition = 'Uninjured'
df = niches[niches.condition==condition][list(niches[col_name].unique()) + [col_name]]
fc = CI.neighborinteractions(df, col_name, mode='pct', save=[outdir,condition], cmap='bwr', method='ward', vmin=0.25, vmax=0.75)

# plot network graph of cell-cell interactions 
g, pos = CI.networkmap(fc, edgemax=20, layout='circle', sort=False, order=circle_order, node_size=[30]*len(circle_order), node_colors=['black']*len(circle_order), filter=0.6, r=200, arc=0.03, save=[outdir,condition+'_circle'],self_edge=False)

In [ ]:
from scipy import stats

In [ ]:
stats.ranksums(df[df['subset_2']=='T cells-CD8']['APCs'],df[df['subset_2']!='T cells-CD8']['APCs'])